# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

Hint: beware of missing rows of data.
The source is missing a few months!

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

seed = 42

df = pd.read_csv('strawberry-prices.tsv', delimiter='\t')
df['month'] = pd.to_datetime(df['month'])
df = df.sort_values(by='month')
df

,month,price
0,2020-01-01,4.049
1,2020-02-01,3.625
2,2020-03-01,3.377
3,2020-05-01,3.126
4,2020-06-01,2.926
...,...,...
63,2025-06-01,3.190
64,2025-07-01,3.187
65,2025-08-01,3.430
66,2025-09-01,3.645


In [2]:
df[df.isna().any(axis=1)]

,month,price


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   month   68 non-null     datetime64[ns]
 1   price   68 non-null     float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.2 KB


In [4]:
df_copy = df.copy()
df_copy = df_copy.set_index('month')
df_copy.index = pd.to_datetime(df_copy.index)

# return which dates are missing in the sequence (frequency is monthly from 1st day of month)
print(pd.date_range(start='2020-01-01', end='2025-12-01', freq='MS').difference(df_copy.index))

DatetimeIndex(['2020-04-01', '2021-12-01', '2025-10-01', '2025-11-01'], dtype='datetime64[ns]', freq=None)


In [5]:
train = pd.DataFrame({
    'month': pd.date_range(start='2020-01-01', end='2024-12-01', freq='MS')
})

train = pd.merge(train, df, on='month', how='left')
# train

test = pd.DataFrame({
    'month': pd.date_range(start='2025-01-01', end='2025-12-01', freq='MS')
})

test = pd.merge(test, df, on='month', how='left')
test


,month,price
0,2025-01-01,4.584
1,2025-02-01,4.077
2,2025-03-01,3.369
3,2025-04-01,3.518
4,2025-05-01,3.416
5,2025-06-01,3.190
6,2025-07-01,3.187
7,2025-08-01,3.430
8,2025-09-01,3.645
9,2025-10-01,NaN


In [6]:
train['price'] = train['price'].interpolate(method='linear')
test['price'] = test['price'].interpolate(method='linear')

In [7]:
def create_features(df):
    df = df.copy()
    df = df.set_index('month')
    df.index = pd.to_datetime(df.index)

    # df['day_of_week'] = df.index.day_of_week
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    # df['day_of_year'] = df.index.day_of_year
    df['year'] = df.index.year
    return df

train = create_features(train)
test = create_features(test)

FEATURES = ['quarter', 'month',  'year']
TARGET = 'price'

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [8]:
import xgboost as xgb

reg = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.01, random_state=seed)
reg.fit(X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=50)

[0]	validation_0-rmse:0.55878	validation_1-rmse:0.57583
[50]	validation_0-rmse:0.39500	validation_1-rmse:0.40362
[100]	validation_0-rmse:0.28629	validation_1-rmse:0.29274
[150]	validation_0-rmse:0.21173	validation_1-rmse:0.22826
[200]	validation_0-rmse:0.16073	validation_1-rmse:0.19658
[250]	validation_0-rmse:0.12497	validation_1-rmse:0.18665
[300]	validation_0-rmse:0.09926	validation_1-rmse:0.18901
[350]	validation_0-rmse:0.08069	validation_1-rmse:0.19451
[400]	validation_0-rmse:0.06650	validation_1-rmse:0.20080
[450]	validation_0-rmse:0.05562	validation_1-rmse:0.20697
[500]	validation_0-rmse:0.04787	validation_1-rmse:0.21192
[550]	validation_0-rmse:0.04209	validation_1-rmse:0.21620
[600]	validation_0-rmse:0.03627	validation_1-rmse:0.21977
[650]	validation_0-rmse:0.03129	validation_1-rmse:0.22211
[700]	validation_0-rmse:0.02786	validation_1-rmse:0.22363
[750]	validation_0-rmse:0.02509	validation_1-rmse:0.22468
[800]	validation_0-rmse:0.02306	validation_1-rmse:0.22542
[850]	validation_

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [9]:
test['prediction'] = reg.predict(X_test)
test = test[['price', 'prediction']]
test['month'] = test.index
test.reset_index(drop=True, inplace=True)
test

,price,prediction,month
0,4.584000,5.042923,2025-01-01
1,4.077000,4.280049,2025-02-01
2,3.369000,3.718337,2025-03-01
3,3.518000,3.589300,2025-04-01
4,3.416000,3.224733,2025-05-01
5,3.190000,3.002942,2025-06-01
6,3.187000,3.111900,2025-07-01
7,3.430000,3.351078,2025-08-01
8,3.645000,3.727358,2025-09-01
9,4.092667,3.724003,2025-10-01


In [10]:
test.to_csv('strawberry-backtest.tsv', sep='\t', index=False)

Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [11]:
import numpy as np

actual = df[df['month'].dt.year == 2025].copy()

eval = pd.merge(
    actual, test, on='month', suffixes=('_actual', '_pred')
)

residuals = eval['price_actual'] - eval['price_pred']

errors = pd.DataFrame([{
    'mean': residuals.mean(),
    'std': residuals.std(ddof=1)
}])

errors

,mean,std
0,0.0,0.0


In [12]:
errors.to_csv('backtest-accuracy.tsv', sep='\t', index=False)

Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [13]:
train = pd.DataFrame({
    'month': pd.date_range(start='2020-01-01', end='2024-12-01', freq='MS')
})

train = pd.merge(train, df, on='month', how='left')
# train

test = pd.DataFrame({
    'month': pd.date_range(start='2025-01-01', end='2025-12-01', freq='MS')
})

test = pd.merge(test, df, on='month', how='left')

train['price'] = train['price'].interpolate(method='linear')
test['price'] = test['price'].interpolate(method='linear')

df_train = pd.concat([train, test])
df_train

,month,price
0,2020-01-01,4.049000
1,2020-02-01,3.625000
2,2020-03-01,3.377000
3,2020-04-01,3.251500
4,2020-05-01,3.126000
...,...,...
7,2025-08-01,3.430000
8,2025-09-01,3.645000
9,2025-10-01,4.092667
10,2025-11-01,4.540333


In [14]:
train = create_features(df_train)

FEATURES = ['quarter', 'month', 'year']
TARGET = 'price'

X_train = train[FEATURES]
y_train = train[TARGET]

reg = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.01, random_state=seed)
reg.fit(X_train, y_train,
        eval_set=[(X_train, y_train)],
        verbose=50)

forecast_dates = pd.date_range(start='2026-01-01', end='2026-12-01', freq='MS')

df_forecast = pd.DataFrame()
df_forecast['month'] = forecast_dates
df_forecast['price'] = np.nan


extended_df = pd.concat([train[['price']], df_forecast])
extended_df = create_features(extended_df)

X_forecast = extended_df.loc[forecast_dates, FEATURES]

extended_df.loc[forecast_dates, TARGET] = reg.predict(X_forecast)

forecast = extended_df.loc[forecast_dates, TARGET]
forecast = forecast.reset_index()
forecast.columns = ['month', 'price']
forecast

[0]	validation_0-rmse:0.56155
[50]	validation_0-rmse:0.39102
[100]	validation_0-rmse:0.27992
[150]	validation_0-rmse:0.20528
[200]	validation_0-rmse:0.15485
[250]	validation_0-rmse:0.12091
[300]	validation_0-rmse:0.09762
[350]	validation_0-rmse:0.08226
[400]	validation_0-rmse:0.07074
[450]	validation_0-rmse:0.06254
[500]	validation_0-rmse:0.05574
[550]	validation_0-rmse:0.05194
[600]	validation_0-rmse:0.04346
[650]	validation_0-rmse:0.04031
[700]	validation_0-rmse:0.03785
[750]	validation_0-rmse:0.03488
[800]	validation_0-rmse:0.02936
[850]	validation_0-rmse:0.02580
[900]	validation_0-rmse:0.02262
[950]	validation_0-rmse:0.02018
[999]	validation_0-rmse:0.01861


,month,price
0,2026-01-01,4.582495
1,2026-02-01,4.076678
2,2026-03-01,3.392399
3,2026-04-01,3.518736
4,2026-05-01,3.392388
5,2026-06-01,3.192942
6,2026-07-01,3.185744
7,2026-08-01,3.422779
8,2026-09-01,3.671136
9,2026-10-01,4.078383


In [15]:
forecast.to_csv('strawberry-forecast.tsv', sep='\t', index=False)

Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [16]:
forecast

,month,price
0,2026-01-01,4.582495
1,2026-02-01,4.076678
2,2026-03-01,3.392399
3,2026-04-01,3.518736
4,2026-05-01,3.392388
5,2026-06-01,3.192942
6,2026-07-01,3.185744
7,2026-08-01,3.422779
8,2026-09-01,3.671136
9,2026-10-01,4.078383


In [17]:
import itertools

BUDGET = 1000000            # $1M investment to cover all costs
FREEZE_COST = 0.20          # $0.20 per pint to freeze
STORAGE_PER_MONTH = 0.10    # $0.10 per pint per month to freeze
DISCOUNT = 0.9              # 10% price discount from selling previously frozen strawberries

results = []

# check every combination where selling is after buying
for (buy_idx, buy_sample), (sell_idx, sell_sample) in itertools.combinations(forecast.iterrows(), 2):

    # number of months holding strawberries
    m = sell_sample['month'].month - buy_sample['month'].month

    # costs and revenue
    pint_cost = buy_sample['price'] + FREEZE_COST + (STORAGE_PER_MONTH * m)
    revenue = sell_sample['price'] * DISCOUNT
    pint_profit = revenue - pint_cost

    # how many pints can we afford
    pints_purchased = int(BUDGET // pint_cost)
    expected_profit = pints_purchased * pint_profit

    results.append({
        'buy_month': buy_sample['month'].strftime('%Y-%m-%d'),
        'sell_month': sell_sample['month'].strftime('%Y-%m-%d'),
        'pints_purchased': pints_purchased,
        'expected_profit': expected_profit
    })

results = pd.DataFrame(results)

maximum_expected_profit = results.sort_values(by='expected_profit', ascending=False).iloc[0]

print('--- Top Strategies --- ')
print(results.sort_values(by='expected_profit', ascending=False).head())

print('\n--- Maximum Profit Strategy ---')
print(maximum_expected_profit)

--- Top Strategies --- 
     buy_month  sell_month  pints_purchased  expected_profit
55  2026-07-01  2026-12-01           257350    154511.091828
50  2026-06-01  2026-12-01           250441    123516.179655
59  2026-08-01  2026-12-01           248584    115183.390238
54  2026-07-01  2026-11-01           264148     79396.629916
62  2026-09-01  2026-12-01           239742     75518.789445

--- Maximum Profit Strategy ---
buy_month             2026-07-01
sell_month            2026-12-01
pints_purchased           257350
expected_profit    154511.091828
Name: 55, dtype: object


In [18]:
results.to_csv('timings.tsv', sep='\t', index=False)

Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [19]:
best_profit = maximum_expected_profit['expected_profit']
std = errors['std'][0]

check = pd.DataFrame([{
    'best_profit': best_profit,
    'one_std_profit': best_profit * std
}])

check

,best_profit,one_std_profit
0,154511.091828,0.0


In [20]:
check.to_csv('check.tsv', sep='\t', index=False)

Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.